In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


import pandas as pd
from data.stock_returns import fetch_stock_data, cache_all_stock_data, construct_forward_backward_returns_adtv
from data.data_utils import load_raw_stock_transactions, load_feature_set, load_raw_congressional_transactions, load_standardized_feature_set
import yfinance as yf
from sklearn.ensemble import IsolationForest
from src.models.isolation_forest import run_isolation_forest, get_isolation_model
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
from src.models.deep_svdd import get_anomaly_scores, run_deep_svdd

In [ ]:
df = load_feature_set()

In [ ]:
s_df = df.xs("Kelly Loeffler", level="senator").loc[pd.Timestamp(2020, 1, 24):pd.Timestamp(2020, 2, 14)]
s_df["returns_after_30d"] * s_df["direction"]

date        ticker
2020-01-24  REZI      0.239016
2020-01-27  REZI      0.214418
2020-01-28  REZI      0.307143
2020-01-29  CMCSA     0.193444
            KEYS      0.139034
            ROST      0.200577
2020-01-30  AZO       0.067966
2020-01-31  AZO       0.195310
            ROST      0.338999
            TJX       0.302139
2020-02-05  HON       0.323890
2020-02-06  AZO       0.314579
            CMCSA     0.251794
            TCEHY     0.119992
2020-02-07  FNKO      0.577236
            ROKU      0.280000
            TCEHY     0.090695
2020-02-10  AZO       0.232785
            FNKO      0.533800
2020-02-11  FNKO      0.532203
            ORLY      0.233066
2020-02-12  EEFT      0.366633
            FNKO      0.509229
2020-02-14  ORCL     -0.092122
            XOM       0.381698
dtype: float64

In [ ]:
s_df = df.xs("David Perdue", level="senator").loc[pd.Timestamp(2020, 1, 24):pd.Timestamp(2020, 3, 2)]
s_df["returns_after_30d"] * s_df["direction"]

date        ticker
2020-01-24  BAC       0.341982
            DD       -0.404368
2020-01-27  GPK       0.156516
2020-01-28  AAPL      0.130965
2020-01-29  AXTA      0.360726
            GLW      -0.277954
2020-01-30  BAC       0.273771
2020-02-07  HBI      -0.360944
2020-02-10  ET        0.606235
            HBI      -0.345227
            TRGP      0.773190
            WMB       0.381668
2020-02-11  AAPL      0.231814
2020-02-13  ET        0.627635
2020-02-19  ALB       0.382774
            ENB      -0.351680
            LNG       0.421861
            MPLX      0.518987
            TRGP      0.840595
            WMB       0.365598
2020-02-20  OKE       0.720975
            T         0.255115
            WES       0.806187
2020-02-21  T         0.287678
2020-02-24  CZR       0.777266
            DAL      -0.588420
            T         0.226281
            TRP      -0.193791
2020-02-25  CZR       0.728335
            DAL      -0.562782
            DVN      -0.504825
            GPK     

In [ ]:
s_df = df.xs("James M. Inhofe", level="senator").loc[pd.Timestamp(2020, 1, 24):pd.Timestamp(2020, 3, 2)]
s_df["returns_after_30d"] * s_df["direction"]

date        ticker
2020-01-27  AAPL      0.074228
            DHR       0.104777
            INTU      0.049690
            PYPL      0.047726
dtype: float64

In [ ]:
s_df = df.xs("David Perdue", level="senator")
s_df = s_df.xs("BWXT", level="ticker")
(s_df['amount'] * s_df["returns_after_30d"]).loc[pd.Timestamp(2018, 1, 1):pd.Timestamp(2020, 1, 1)].sum()

np.float64(29085.508401490988)

In [27]:
(df["ticker"] == "SPY").sum()

np.int64(22)

In [20]:
df = load_raw_stock_transactions()
price_df = cache_all_stock_data(df)[0]
spy_price = price_df.xs("SPY", level="ticker")
price_df = price_df.loc[~price_df.index.isin(["SPY"], level="ticker")]

price_df = price_df.join(spy_price, rsuffix="_spy", on="date", how="left")

Data is cached -- reading /Users/declannelson/Desktop/code/congressional_stock_picks/data/returns_cache/1100f11ed8727b728707f8fd450d7ed8/df_beta_adj.parquet
reading failed tickers


In [24]:
price_df.sort_index().groupby(["ticker"]).corr()

Price                  Close      High       Low      Open    Volume  \
ticker  Price                                                          
0QZI.IL Close       1.000000  0.999146  0.998517  0.999397  0.143932   
        High        0.999146  1.000000  0.997727  0.999149  0.148243   
        Low         0.998517  0.997727  1.000000  0.998411  0.138815   
        Open        0.999397  0.999149  0.998411  1.000000  0.143792   
        Volume      0.143932  0.148243  0.138815  0.143792  1.000000   
...                      ...       ...       ...       ...       ...   
ZTS     Close_spy   0.981717  0.981315  0.981864  0.981550 -0.332525   
        High_spy    0.982074  0.981907  0.982296  0.982143 -0.331259   
        Low_spy     0.981110  0.980772  0.981474  0.981142 -0.333692   
        Open_spy    0.981506  0.981350  0.981882  0.981738 -0.332448   
        Volume_spy -0.242244 -0.236941 -0.246320 -0.240730  0.289976   

Price               Close_spy  High_spy   Low_spy  Open_spy  Volume_spy  
ticker  Price                                                            
0QZI.IL Close        0.824678  0.823614  0.825813  0.824859   -0.202722  
        High         0.826725  0.825861  0.827810  0.827076   -0.196720  
        Low          0.822295  0.821143  0.823531  0.822530   -0.205686  
        Open         0.824491  0.823561  0.825694  0.824927   -0.200469  
        Volume       0.323878  0.325278  0.322504  0.324239    0.001773  
...                       ...       ...       ...       ...         ...  
ZTS     Close_spy    1.000000  0.999795  0.999820  0.999675   -0.279323  
        High_spy     0.999795  1.000000  0.999696  0.999855   -0.269622  
        Low_spy      0.999820  0.999696  1.000000  0.999814   -0.286218  
        Open_spy     0.999675  0.999855  0.999814  1.000000   -0.276444  
        Volume_spy  -0.279323 -0.269622 -0.286218 -0.276444    1.000000  

[8250 rows x 10 columns]